# Silver Layer — Clean & Enrich Construction Data
Parses dates, derives schedule variance bands, cost overrun flags, and task completion rates.

In [ ]:
from pyspark.sql.functions import (
    col, to_date, datediff, when, lit, round as spark_round,
    current_timestamp, coalesce
)

df_subs  = spark.read.format('delta').table('bronze_subcontractors')
df_proj  = spark.read.format('delta').table('bronze_projects')
df_tasks = spark.read.format('delta').table('bronze_tasks')
df_costs = spark.read.format('delta').table('bronze_cost_ledger')

In [ ]:
# Silver subcontractors — cast rating
silver_subs = (
    df_subs
    .withColumn('rating', col('rating').cast('double'))
    .withColumn('rating_band',
        when(col('rating') >= 4.5, 'Excellent')
        .when(col('rating') >= 3.5, 'Good')
        .when(col('rating') >= 2.5, 'Fair')
        .otherwise('Poor')
    )
    .withColumn('silver_timestamp', current_timestamp())
)
silver_subs.write.mode('overwrite').format('delta').saveAsTable('silver_subcontractors')
print(f'Silver subcontractors: {spark.table("silver_subcontractors").count()} rows')

In [ ]:
# Silver projects — parse dates, derive variance bands
silver_proj = (
    df_proj
    .withColumn('planned_start_date',  to_date('planned_start_date'))
    .withColumn('planned_end_date',    to_date('planned_end_date'))
    .withColumn('actual_start_date',   to_date('actual_start_date'))
    .withColumn('forecast_end_date',   to_date('forecast_end_date'))
    .withColumn('planned_duration_days',
        datediff(col('planned_end_date'), col('planned_start_date'))
    )
    .withColumn('schedule_variance_days', col('schedule_variance_days').cast('int'))
    .withColumn('cost_variance_pct',      col('cost_variance_pct').cast('double'))
    .withColumn('budget',                 col('budget').cast('double'))
    .withColumn('schedule_risk_band',
        when(col('schedule_variance_days') > 60, 'Critical')
        .when(col('schedule_variance_days') > 30, 'High')
        .when(col('schedule_variance_days') > 10, 'Medium')
        .when(col('schedule_variance_days') >= 0, 'On Track')
        .otherwise('Ahead')
    )
    .withColumn('cost_risk_band',
        when(col('cost_variance_pct') > 20, 'Critical')
        .when(col('cost_variance_pct') > 10, 'High')
        .when(col('cost_variance_pct') > 0,  'Over Budget')
        .otherwise('On Budget')
    )
    .withColumn('silver_timestamp', current_timestamp())
)
silver_proj.write.mode('overwrite').format('delta').saveAsTable('silver_projects')
print(f'Silver projects: {spark.table("silver_projects").count()} rows')

In [ ]:
# Silver tasks — parse dates, derive delay flag
silver_tasks = (
    df_tasks
    .withColumn('planned_start_date',  to_date('planned_start_date'))
    .withColumn('planned_end_date',    to_date('planned_end_date'))
    .withColumn('actual_start_date',   to_date('actual_start_date'))
    .withColumn('forecast_end_date',   to_date('forecast_end_date'))
    .withColumn('schedule_variance_days', col('schedule_variance_days').cast('int'))
    .withColumn('pct_complete',           col('pct_complete').cast('double'))
    .withColumn('is_delayed',
        when(col('schedule_variance_days') > 0, 1).otherwise(0)
    )
    .withColumn('is_completed',
        when(col('status') == 'Completed', 1).otherwise(0)
    )
    .withColumn('silver_timestamp', current_timestamp())
)
silver_tasks.write.mode('overwrite').format('delta').saveAsTable('silver_tasks')
print(f'Silver tasks: {spark.table("silver_tasks").count()} rows')

In [ ]:
# Silver cost ledger — cast numerics, derive overrun flag
silver_costs = (
    df_costs
    .withColumn('entry_date',          to_date('entry_date'))
    .withColumn('planned_cost',        col('planned_cost').cast('double'))
    .withColumn('actual_cost',         col('actual_cost').cast('double'))
    .withColumn('cost_variance',       col('cost_variance').cast('double'))
    .withColumn('cost_variance_pct',   col('cost_variance_pct').cast('double'))
    .withColumn('is_overrun',
        when(col('cost_variance') > 0, 1).otherwise(0)
    )
    .withColumn('overrun_band',
        when(col('cost_variance_pct') > 25, 'Severe')
        .when(col('cost_variance_pct') > 10, 'Significant')
        .when(col('cost_variance_pct') > 0,  'Minor')
        .otherwise('Under Budget')
    )
    .withColumn('silver_timestamp', current_timestamp())
)
silver_costs.write.mode('overwrite').format('delta').saveAsTable('silver_cost_ledger')
print(f'Silver cost ledger: {spark.table("silver_cost_ledger").count()} rows')

In [ ]:
print('Silver transformation complete.')
print(f'  Subcontractors : {silver_subs.count():>7,}')
print(f'  Projects       : {silver_proj.count():>7,}')
print(f'  Tasks          : {silver_tasks.count():>7,}')
print(f'  Cost Ledger    : {silver_costs.count():>7,}')